# INH2
## Dependencies

In [2]:
import os
import tqdm
import shutil
import numpy as np
import scipy.io
import pandas as pd
import pickle
import h5py

import logging

In [3]:
def load_mocap_data(file_path):
    mat = scipy.io.loadmat(file_path)
    exp_name = [m for m in mat.keys() if m[:2] != '__'][0]  ## TOCHANGE
    data = np.moveaxis(mat[exp_name][0, 0]['Trajectories'][0, 0]['Labeled']['Data'][0, 0], 2, 0)
    keypoints = [label[0].replace('coordinate', 'coord') for label in
                    mat[exp_name][0, 0]['Trajectories'][0, 0]['Labeled']['Labels'][0, 0][0]]

    # Very important. Make sure the keypoints are always in the same order even if not saved so in the original files
    new_order = np.argsort(keypoints)
    keypoints = [keypoints[n] for n in new_order]
    data = data[:, new_order, :]

    return data, keypoints

In [6]:
# check file names
file_path = "/home/group-cvg/cvg-rose/bogna_mocap/INH2B_CLB"
files = os.listdir(file_path)
files.remove('._.DS_Store')
print(len(files))
files

72


['INH2B_S21_M9_MC11_CLB_13_03_2020_proc_bij_15_04_2020_C.mat',
 'INH2B_S44_M8_MC10_CLB_19_03_2020_proc_bij_21_04_2020_B.mat',
 'INH2B_S34_M10_MC11_CLB_16_03_2020_proc_bij_16_04_2020_A.mat',
 'INH2B_S7_M7_MC10_CLB_10_03_2020_proc_bij_13_04_2020_F.mat',
 'INH2B_S50_M2_MC9_CLB_23_03_2020_proc_bij_22_04_2020_A.mat',
 'INH2B_S52_M4_MC9_CLB_23_03_2020_proc_bij_22_04_2020_E.mat',
 'INH2B_S11_M11_MC11_CLB_10_03_2020_proc_bij_14_04_2020_B.mat',
 'INH2B_S59_M11_MC11_CLB_23_03_2020_proc_bij_24_04_2020_D.mat',
 'INH2B_S8_M8_MC10_CLB_10_03_2020_proc_bij_13_04_2020_E.mat',
 'INH2B_S68_M8_MC10_CLB_26_03_2020_proc_bij_24_04_2020_F.mat',
 'INH2B_S71_M11_MC11_CLB_26_03_2020_proc_bij_24_04_2020_C.mat',
 'INH2B_S57_M9_MC11_CLB_23_03_2020_proc_bij_23_04_2020_F.mat',
 'INH2B_S56_M8_MC10_CLB_23_03_2020_proc_bij_23_04_2020_A.mat',
 'INH2B_S33_M9_MC11_CLB_16_03_2020_proc_bij_16_04_2020_B.mat',
 'INH2B_S36_M12_MC11_CLB_16_03_2020_proc_bij_16_04_2020_E.mat',
 'INH2B_S51_M3_MC9_CLB_23_03_2020_proc_bij_22_04_2020_

## Process

In [7]:
data_INH2_CLB = {}

for file_name in files:
    file = os.path.join(file_path, file_name)
    
    if os.path.isfile(file):
        data, keypoints = load_mocap_data(file)
        key_name = str(file_name.split("_")[0] + "_" + file_name.split("_")[1]) + "_CLB"
        data_INH2_CLB[key_name] ={"label":None, "keypoints":None}
        data_INH2_CLB[key_name]["label"] = file_name.split(".")[-2][-1]
        data_INH2_CLB[key_name]["keypoints"] = keypoints
        # 1. Discard last dimension
        data = data[ : , : , :3]
        
        data = data[1500: ]
        
        data_INH2_CLB[key_name]["data"] = data

In [10]:
# The correct keypoint 
keypoints_name = ['left_ankle',  'left_back',  'left_coord',  'left_hip',  'left_knee', 
                  'right_ankle', 'right_back', 'right_coord', 'right_hip', 'right_knee']

for key, value in data_INH2_CLB.items():
    
    # Create an empty array to store
    arr = np.full((18000, 10, 3), np.nan)
    i = 0  # index of correct keypoint
    j = 0  # index of actual keypoint
    while i < 10:
        if keypoints_name[i] == value["keypoints"][i]:
            arr[:, i, :] = value["data"][:, j, :]
            i += 1 
            j += 1
        else:
            value["keypoints"].insert(i, None)
            i += 1
    
    value["data"] = arr 

In [11]:
import pickle

with open('../../data/data_INH2_CLB.pkl', 'wb') as file:
    pickle.dump(data_INH2_CLB, file)